In [2]:
import requests
teams = requests.get("https://api-web.nhle.com/v1/standings/2026-04-17")

In [5]:
teams

<Response [200]>

In [10]:
data: dict = teams.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['wildCardIndicator', 'standingsDateTimeUtc', 'standings'])


In [7]:
type(data)

dict

In [8]:
data

{'wildCardIndicator': True,
 'standingsDateTimeUtc': '2026-09-20T15:35:30Z',
 'standings': [{'clinchIndicator': 'p',
   'conferenceAbbrev': 'W',
   'conferenceHomeSequence': 1,
   'conferenceL10Sequence': 2,
   'conferenceName': 'Western',
   'conferenceRoadSequence': 1,
   'conferenceSequence': 1,
   'date': '2026-04-17',
   'divisionAbbrev': 'C',
   'divisionHomeSequence': 1,
   'divisionL10Sequence': 1,
   'divisionName': 'Central',
   'divisionRoadSequence': 1,
   'divisionSequence': 1,
   'gameTypeId': 2,
   'gamesPlayed': 82,
   'goalDifferential': 99,
   'goalDifferentialPctg': 1.207317,
   'goalAgainst': 203,
   'goalFor': 302,
   'goalsForPctg': 3.682927,
   'homeGamesPlayed': 41,
   'homeGoalDifferential': 49,
   'homeGoalsAgainst': 108,
   'homeGoalsFor': 157,
   'homeLosses': 9,
   'homeOtLosses': 6,
   'homePoints': 58,
   'homeRegulationPlusOtWins': 25,
   'homeRegulationWins': 25,
   'homeTies': 0,
   'homeWins': 26,
   'l10GamesPlayed': 10,
   'l10GoalDifferential': 14,

In [9]:
data['standings']

[{'clinchIndicator': 'p',
  'conferenceAbbrev': 'W',
  'conferenceHomeSequence': 1,
  'conferenceL10Sequence': 2,
  'conferenceName': 'Western',
  'conferenceRoadSequence': 1,
  'conferenceSequence': 1,
  'date': '2026-04-17',
  'divisionAbbrev': 'C',
  'divisionHomeSequence': 1,
  'divisionL10Sequence': 1,
  'divisionName': 'Central',
  'divisionRoadSequence': 1,
  'divisionSequence': 1,
  'gameTypeId': 2,
  'gamesPlayed': 82,
  'goalDifferential': 99,
  'goalDifferentialPctg': 1.207317,
  'goalAgainst': 203,
  'goalFor': 302,
  'goalsForPctg': 3.682927,
  'homeGamesPlayed': 41,
  'homeGoalDifferential': 49,
  'homeGoalsAgainst': 108,
  'homeGoalsFor': 157,
  'homeLosses': 9,
  'homeOtLosses': 6,
  'homePoints': 58,
  'homeRegulationPlusOtWins': 25,
  'homeRegulationWins': 25,
  'homeTies': 0,
  'homeWins': 26,
  'l10GamesPlayed': 10,
  'l10GoalDifferential': 14,
  'l10GoalsAgainst': 20,
  'l10GoalsFor': 34,
  'l10Losses': 2,
  'l10OtLosses': 1,
  'l10Points': 15,
  'l10RegulationPlus

In [18]:
records = []

for team_id, team in enumerate(data['standings'], start=1):
    team_abbrev = team['teamAbbrev']['default']
    team_name = team['teamName']['default']
    conf_name = team['conferenceName']
    div_name = team['divisionName']
    logo_url = team['teamLogo']

    records.append(dict(
        team_id=team_id,
        team_abbrev=team_abbrev,
        team_name=team_name,
        conference_name=conf_name,
        division_name=div_name,
        logo_url=logo_url
    ))

print(f'Total teams: {len(records)}')
records

Total teams: 32


[{'team_id': 1,
  'team_abbrev': 'COL',
  'team_name': 'Colorado Avalanche',
  'conference_name': 'Western',
  'division_name': 'Central',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg'},
 {'team_id': 2,
  'team_abbrev': 'CAR',
  'team_name': 'Carolina Hurricanes',
  'conference_name': 'Eastern',
  'division_name': 'Metropolitan',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/CAR_light.svg'},
 {'team_id': 3,
  'team_abbrev': 'DAL',
  'team_name': 'Dallas Stars',
  'conference_name': 'Western',
  'division_name': 'Central',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/DAL_light.svg'},
 {'team_id': 4,
  'team_abbrev': 'BUF',
  'team_name': 'Buffalo Sabres',
  'conference_name': 'Eastern',
  'division_name': 'Atlantic',
  'logo_url': 'https://assets.nhle.com/logos/nhl/svg/BUF_light.svg'},
 {'team_id': 5,
  'team_abbrev': 'TBL',
  'team_name': 'Tampa Bay Lightning',
  'conference_name': 'Eastern',
  'division_name': 'Atlantic',
  'logo_url': 'https://ass

In [3]:
import mysql.connector

connection = mysql.connector.connect(
    host='localhost',
    port=3306,
    database='NHL_ANALYTICS_PROJECT',
    user='root',
    password="Omsivaya@20"
)

In [21]:
if connection.is_connected():
        cursor = connection.cursor()

In [22]:
%pip install mysql-connector-python


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS teams (
    team_id INT AUTO_INCREMENT PRIMARY KEY,
    team_abbrev VARCHAR(10) UNIQUE,
    team_name VARCHAR(100) NOT NULL,
    conference_name VARCHAR(100),
    division_name VARCHAR(100),
    logo_url TEXT
)
""")

connection.commit()
print('Teams table created successfully.')

Teams table created successfully.


In [31]:
insert_query = """
INSERT INTO teams (
    team_abbrev,
    team_name,
    conference_name,
    division_name,
    logo_url
)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_name = VALUES(team_name),
    conference_name = VALUES(conference_name),
    division_name = VALUES(division_name),
    logo_url = VALUES(logo_url)
"""

for i in records:
    team_values = (
        i['team_abbrev'],
        i['team_name'],
        i['conference_name'],
        i['division_name'],
        i['logo_url']
    )

    cursor.execute(insert_query, team_values)

connection.commit()
print(f"{len(records)} teams inserted or updated successfully.")

32 teams inserted or updated successfully.


In [35]:
import pandas as pd
df = pd.read_sql_query("select * from teams", connection)
df

/var/folders/f7/2dmww_657j51jk3c1xsbw30h0000gn/T/ipykernel_6132/2365296734.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("select * from teams", connection)


,team_id,team_abbrev,team_name,conference_name,division_name,logo_url
0,1,COL,Colorado Avalanche,Western,Central,https://assets.nhle.com/logos/nhl/svg/COL_ligh...
1,2,CAR,Carolina Hurricanes,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...
2,3,DAL,Dallas Stars,Western,Central,https://assets.nhle.com/logos/nhl/svg/DAL_ligh...
3,4,BUF,Buffalo Sabres,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BUF_ligh...
4,5,TBL,Tampa Bay Lightning,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/TBL_ligh...
5,6,MTL,Montréal Canadiens,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/MTL_ligh...
6,7,MIN,Minnesota Wild,Western,Central,https://assets.nhle.com/logos/nhl/svg/MIN_ligh...
7,8,BOS,Boston Bruins,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BOS_ligh...
8,9,OTT,Ottawa Senators,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/OTT_ligh...
9,10,PIT,Pittsburgh Penguins,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/PIT_ligh...


In [ ]:
#Last 3 tables

df_games = pd.read_json()

In [36]:
# Push it into SQL
!pip install sqlalchemy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 9.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [37]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.9 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
Note: you may need to restart the kernel to use updated packages.


In [38]:
import pymysql

from sqlalchemy import create_engine

engine = create_engine('mysql+pymysql://root:Omsivaya@20@localhost/NHL_ANALYTICS_PROJECT')

In [39]:
df_games = pd.read_json('/Users/nareshselvaraj/pandas course/NHL Analytics Hub Guvi Project/game_stats.json')
df_games

,stat_id,game_id,player_id,team_id,goals,assists,points,shots_on_goal,penalty_min,toi,plus_minus
0,1,2025010051,8480448,1,0,0,0,0,0,14:28,0
1,2,2025010051,8479525,1,0,0,0,0,0,18:48,-1
2,3,2025010051,8477492,1,0,1,1,0,0,18:57,1
3,6,2025010051,8477476,1,1,1,2,0,0,20:44,2
4,10,2025010051,8482947,1,1,0,1,0,0,12:48,1
...,...,...,...,...,...,...,...,...,...,...,...
47403,53923,2025021300,8476879,20,0,0,0,0,0,18:25,-1
47404,53924,2025021300,8476441,20,0,0,0,0,0,18:48,0
47405,53925,2025021300,8474563,20,0,1,1,0,0,22:20,1
47406,53926,2025021300,8479998,20,0,0,0,0,0,19:50,2


In [42]:
df_games.to_sql('game_stats',if_exists='replace', con=engine)

47408

In [44]:
import requests

standings_response = requests.get(
    "https://api-web.nhle.com/v1/standings/now",
    timeout=30
)
standings_response.raise_for_status()
standings_payload = standings_response.json()
standings_data = standings_payload['standings']

print(f"Standings records: {len(standings_data)}")
print(sorted(standings_data[0].keys()))
print([(team['teamAbbrev'], team['streakCode'], team['streakCount']) for team in standings_data[:5]])

Standings records: 32
['conferenceAbbrev', 'conferenceHomeSequence', 'conferenceL10Sequence', 'conferenceName', 'conferenceRoadSequence', 'conferenceSequence', 'date', 'divisionAbbrev', 'divisionHomeSequence', 'divisionL10Sequence', 'divisionName', 'divisionRoadSequence', 'divisionSequence', 'gameTypeId', 'gamesPlayed', 'goalAgainst', 'goalDifferential', 'goalDifferentialPctg', 'goalFor', 'goalsForPctg', 'homeGamesPlayed', 'homeGoalDifferential', 'homeGoalsAgainst', 'homeGoalsFor', 'homeLosses', 'homeOtLosses', 'homePoints', 'homeRegulationPlusOtWins', 'homeRegulationWins', 'homeTies', 'homeWins', 'l10GamesPlayed', 'l10GoalDifferential', 'l10GoalsAgainst', 'l10GoalsFor', 'l10Losses', 'l10OtLosses', 'l10Points', 'l10RegulationPlusOtWins', 'l10RegulationWins', 'l10Ties', 'l10Wins', 'leagueHomeSequence', 'leagueL10Sequence', 'leagueRoadSequence', 'leagueSequence', 'losses', 'otLosses', 'placeName', 'pointPctg', 'points', 'regulationPlusOtWinPctg', 'regulationPlusOtWins', 'regulationWinPct

In [47]:
standings_cursor = connection.cursor()

standings_cursor.execute("""
CREATE TABLE IF NOT EXISTS standings (
    standing_id INT AUTO_INCREMENT PRIMARY KEY,
    team_id INT NOT NULL,
    season VARCHAR(20) NOT NULL,
    games_played INT NOT NULL,
    wins INT NOT NULL,
    losses INT NOT NULL,
    ot_losses INT NOT NULL,
    points INT NOT NULL,
    goals_for INT NOT NULL,
    goals_against INT NOT NULL,
    home_wins INT NOT NULL,
    away_wins INT NOT NULL,
    streak_type VARCHAR(20),
    streak_count INT,
    UNIQUE KEY uq_standings_team_season (team_id, season),
    CONSTRAINT fk_standings_team
        FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB
""")

standings_cursor.execute("SELECT team_id, team_abbrev FROM teams")
team_ids_by_abbrev = {
    team_abbrev: team_id
    for team_id, team_abbrev in standings_cursor.fetchall()
}

standings_records = []
missing_teams = []

for team in standings_data:
    abbrev_value = team['teamAbbrev']
    team_abbrev = abbrev_value['default'] if isinstance(abbrev_value, dict) else abbrev_value
    team_id = team_ids_by_abbrev.get(team_abbrev)

    if team_id is None:
        missing_teams.append(team_abbrev)
        continue

    standings_records.append((
        team_id,
        str(team['seasonId']),
        team['gamesPlayed'],
        team['wins'],
        team['losses'],
        team['otLosses'],
        team['points'],
        team['goalFor'],
        team['goalAgainst'],
        team['homeWins'],
        team['roadWins'],
        team.get('streakCode'),
        team.get('streakCount')
    ))

if missing_teams:
    raise ValueError(f"Teams missing from teams table: {missing_teams}")

upsert_standings_query = """
INSERT INTO standings (
    team_id, season, games_played, wins, losses, ot_losses, points,
    goals_for, goals_against, home_wins, away_wins, streak_type, streak_count
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    games_played = VALUES(games_played),
    wins = VALUES(wins),
    losses = VALUES(losses),
    ot_losses = VALUES(ot_losses),
    points = VALUES(points),
    goals_for = VALUES(goals_for),
    goals_against = VALUES(goals_against),
    home_wins = VALUES(home_wins),
    away_wins = VALUES(away_wins),
    streak_type = VALUES(streak_type),
    streak_count = VALUES(streak_count)
"""

standings_cursor.executemany(upsert_standings_query, standings_records)
connection.commit()

standings_cursor.execute("""
SELECT season, COUNT(*) AS team_count
FROM standings
GROUP BY season
ORDER BY season
""")
print(f"Loaded {len(standings_records)} standings rows.")
print(standings_cursor.fetchall())

Loaded 32 standings rows.
[('20262027', 32)]


In [49]:
standings_cursor

In [51]:
standings_cursor.execute("""
SELECT
    t.team_abbrev,
    t.team_name,
    s.season,
    s.games_played,
    s.wins,
    s.losses,
    s.ot_losses,
    s.points,
    s.goals_for,
    s.goals_against,
    s.home_wins,
    s.away_wins,
    s.streak_type,
    s.streak_count
FROM standings AS s
JOIN teams AS t ON t.team_id = s.team_id
ORDER BY s.points DESC, s.wins DESC
""")

standings_rows = standings_cursor.fetchall()
standings_df = pd.DataFrame(standings_rows, columns=[column[0] for column in standings_cursor.description])
standings_df

,team_abbrev,team_name,season,games_played,wins,losses,ot_losses,points,goals_for,goals_against,home_wins,away_wins,streak_type,streak_count
0,EDM,Edmonton Oilers,20262027,2,1,0,1,3,14,13,0,1,W,1.0
1,FLA,Florida Panthers,20262027,2,1,0,1,3,4,4,0,1,OT,1.0
2,PIT,Pittsburgh Penguins,20262027,1,1,0,0,2,7,0,0,1,W,1.0
3,UTA,Utah Mammoth,20262027,1,1,0,0,2,6,0,1,0,W,1.0
4,SEA,Seattle Kraken,20262027,1,1,0,0,2,6,1,0,1,W,1.0
5,COL,Colorado Avalanche,20262027,1,1,0,0,2,8,4,1,0,W,1.0
6,CBJ,Columbus Blue Jackets,20262027,1,1,0,0,2,6,3,1,0,W,1.0
7,VGK,Vegas Golden Knights,20262027,1,1,0,0,2,5,2,1,0,W,1.0
8,BOS,Boston Bruins,20262027,1,1,0,0,2,3,0,1,0,W,1.0
9,MIN,Minnesota Wild,20262027,1,1,0,0,2,3,1,0,1,W,1.0


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

players_cursor = connection.cursor()
players_cursor.execute("""
CREATE TABLE IF NOT EXISTS players (
    player_id BIGINT PRIMARY KEY,
    team_id INT NOT NULL,
    first_name VARCHAR(100) NOT NULL,
    last_name VARCHAR(100) NOT NULL,
    position VARCHAR(10),
    jersey_number INT,
    birth_date DATE,
    birth_country VARCHAR(10),
    height_cm REAL,
    weight_kg REAL,
    shoots_catches VARCHAR(5),
    headshot_url TEXT,
    CONSTRAINT fk_players_team
        FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB
""")

players_cursor.execute("SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev")
teams_for_rosters = players_cursor.fetchall()


def fetch_team_roster(team_id, team_abbrev):
    response = requests.get(
        f"https://api-web.nhle.com/v1/roster/{team_abbrev}/current",
        timeout=(5, 10)
    )
    response.raise_for_status()
    roster = response.json()
    position_map = {"L": "LW", "R": "RW"}
    team_records = []

    for group_name in ("forwards", "defensemen", "goalies"):
        for player in roster.get(group_name, []):
            first_name = player.get("firstName")
            last_name = player.get("lastName")
            position_code = player.get("positionCode")

            team_records.append((
                player["id"],
                team_id,
                first_name.get("default") if isinstance(first_name, dict) else first_name,
                last_name.get("default") if isinstance(last_name, dict) else last_name,
                position_map.get(position_code, position_code),
                player.get("sweaterNumber"),
                player.get("birthDate"),
                player.get("birthCountry"),
                player.get("heightInCentimeters"),
                player.get("weightInKilograms"),
                player.get("shootsCatches"),
                player.get("headshot")
            ))

    return team_abbrev, team_records


player_records = []
roster_errors = []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(fetch_team_roster, team_id, team_abbrev): team_abbrev
        for team_id, team_abbrev in teams_for_rosters
    }

    for completed_count, future in enumerate(as_completed(futures), start=1):
        team_abbrev = futures[future]
        try:
            _, team_records = future.result()
            player_records.extend(team_records)
            print(f"Fetched {team_abbrev}: {len(team_records)} players ({completed_count}/{len(futures)} teams)")
        except requests.RequestException as error:
            roster_errors.append((team_abbrev, str(error)))
            print(f"Failed {team_abbrev} ({completed_count}/{len(futures)} teams): {error}")

if roster_errors:
    raise RuntimeError(
        f"Roster fetch failed for {len(roster_errors)} teams; no database rows were written. "
        f"Retry the cell after checking the network: {roster_errors}"
    )

upsert_players_query = """
INSERT INTO players (
    player_id, team_id, first_name, last_name, position, jersey_number,
    birth_date, birth_country, height_cm, weight_kg, shoots_catches, headshot_url
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_id = VALUES(team_id),
    first_name = VALUES(first_name),
    last_name = VALUES(last_name),
    position = VALUES(position),
    jersey_number = VALUES(jersey_number),
    birth_date = VALUES(birth_date),
    birth_country = VALUES(birth_country),
    height_cm = VALUES(height_cm),
    weight_kg = VALUES(weight_kg),
    shoots_catches = VALUES(shoots_catches),
    headshot_url = VALUES(headshot_url)
"""

players_cursor.executemany(upsert_players_query, player_records)
connection.commit()

players_cursor.execute("""
SELECT COUNT(*) AS player_count, COUNT(DISTINCT team_id) AS team_count
FROM players
""")
print(f"Fetched and saved {len(player_records)} player records.")
print("Database totals (players, teams):", players_cursor.fetchone())

In [2]:
import pandas as pd
df_players = pd.read_sql_query("SELECT * FROM players", connection)
df_players

NameError: name 'connection' is not defined

In [4]:
import mysql.connector
import pandas as pd

connection = mysql.connector.connect(
    host='localhost',
    port=3306,
    database='NHL_ANALYTICS_PROJECT',
    user='root',
    password="Omsivaya@20"
)

df_players = pd.read_sql_query("SELECT * FROM players", connection)
df_players


/var/folders/f7/2dmww_657j51jk3c1xsbw30h0000gn/T/ipykernel_16340/2531164442.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_players = pd.read_sql_query("SELECT * FROM players", connection)


,player_id,team_id,first_name,last_name,position,jersey_number,birth_date,birth_country,height_cm,weight_kg,shoots_catches,headshot_url
0,8470613,1,Brent,Burns,D,84.0,1985-03-09,CAN,196.0,103.0,R,https://assets.nhle.com/mugs/nhl/20262027/COL/...
1,8470621,20,Corey,Perry,RW,90.0,1985-05-16,CAN,191.0,95.0,R,https://assets.nhle.com/mugs/nhl/20262027/LAK/...
2,8471214,12,Alex,Ovechkin,LW,8.0,1985-09-17,RUS,191.0,108.0,R,https://assets.nhle.com/mugs/nhl/20262027/WSH/...
3,8471215,10,Evgeni,Malkin,C,71.0,1986-07-31,RUS,196.0,96.0,L,https://assets.nhle.com/mugs/nhl/20262027/PIT/...
4,8471675,10,Sidney,Crosby,C,87.0,1987-08-07,CAN,180.0,93.0,L,https://assets.nhle.com/mugs/nhl/20262027/PIT/...
...,...,...,...,...,...,...,...,...,...,...,...,...
767,8486072,18,Marcus,Nordmark,LW,43.0,2008-05-04,SWE,188.0,82.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
768,8486103,24,Ivar,Stenberg,LW,41.0,2007-09-30,SWE,180.0,84.0,L,https://assets.nhle.com/mugs/nhl/20262027/SJS/...
769,8486161,14,Owen,Michaels,RW,43.0,2002-05-01,USA,185.0,83.0,R,https://assets.nhle.com/mugs/nhl/20262027/EDM/...
770,8486166,7,Viking,Gustafsson Nyberg,D,6.0,2003-09-21,SWE,198.0,102.0,L,https://assets.nhle.com/mugs/nhl/20262027/MIN/...


In [10]:
import json
import time
import requests

json_path = r"/Users/nareshselvaraj/pandas course/NHL Analytics Hub Guvi Project/skater_season_stats.json"

with open(json_path, "r") as json_file:
    skater_data = json.load(json_file)

skater_cursor = connection.cursor()
skater_cursor.execute("SELECT team_id, team_abbrev FROM teams")
team_rows = skater_cursor.fetchall()
db_team_ids = {team_id for team_id, _ in team_rows}
db_team_id_by_abbrev = {team_abbrev: team_id for team_id, team_abbrev in team_rows}

source_team_ids = {int(row["team_id"]) for row in skater_data}
unknown_team_ids = source_team_ids - db_team_ids
if unknown_team_ids:
    raise ValueError(f"JSON contains team IDs missing from teams table: {sorted(unknown_team_ids)}")

skater_cursor.execute("SELECT player_id, position FROM players")
player_positions = {
    int(player_id): position
    for player_id, position in skater_cursor.fetchall()
}

missing_player_ids = sorted({int(row["player_id"]) for row in skater_data} - set(player_positions))
source_team_by_player = {
    int(row["player_id"]): int(row["team_id"])
    for row in skater_data
}


def localized_value(value):
    return value.get("default") if isinstance(value, dict) else value


def fetch_player_details(player_id):
    url = f"https://api-web.nhle.com/v1/player/{player_id}/landing"

    for attempt in range(5):
        try:
            response = requests.get(url, timeout=(5, 15))
            if response.status_code == 429 or response.status_code >= 500:
                retry_after = response.headers.get("Retry-After")
                delay = float(retry_after) if retry_after else min(2 ** attempt, 30)
                print(f"NHL API busy for player {player_id}; retrying in {delay:g}s")
                time.sleep(delay)
                continue

            response.raise_for_status()
            player = response.json()
            abbrev = player.get("currentTeamAbbrev")
            team_id = db_team_id_by_abbrev.get(abbrev, source_team_by_player[player_id])

            return (
                player_id,
                team_id,
                localized_value(player.get("firstName")) or "Unknown",
                localized_value(player.get("lastName")) or "Unknown",
                localized_value(player.get("position")),
                player.get("sweaterNumber"),
                player.get("birthDate"),
                player.get("birthCountry"),
                player.get("heightInCentimeters"),
                player.get("weightInKilograms"),
                player.get("shootsCatches"),
                player.get("headshot")
            )
        except requests.RequestException:
            if attempt == 4:
                raise
            time.sleep(min(2 ** attempt, 30))

    raise RuntimeError(f"Could not fetch player {player_id} after retries")


supplemental_player_records = []
for player_number, player_id in enumerate(missing_player_ids, start=1):
    supplemental_player_records.append(fetch_player_details(player_id))
    if player_number % 10 == 0 or player_number == len(missing_player_ids):
        print(f"Fetched player details: {player_number}/{len(missing_player_ids)}")
    time.sleep(0.25)

skater_cursor.execute("""
CREATE TABLE IF NOT EXISTS skater_season_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    player_id BIGINT NOT NULL,
    season VARCHAR(20) NOT NULL,
    team_id INT NOT NULL,
    games_played INT,
    goals INT,
    assists INT,
    points INT,
    plus_minus INT,
    penalty_min INT,
    shots INT,
    avg_toi VARCHAR(10),
    UNIQUE KEY uq_skater_player_season_team (player_id, season, team_id),
    CONSTRAINT fk_skater_stats_player
        FOREIGN KEY (player_id) REFERENCES players(player_id),
    CONSTRAINT fk_skater_stats_team
        FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB
""")

upsert_player_query = """
INSERT INTO players (
    player_id, team_id, first_name, last_name, position, jersey_number,
    birth_date, birth_country, height_cm, weight_kg, shoots_catches, headshot_url
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    team_id = VALUES(team_id),
    first_name = VALUES(first_name),
    last_name = VALUES(last_name),
    position = VALUES(position),
    jersey_number = VALUES(jersey_number),
    birth_date = VALUES(birth_date),
    birth_country = VALUES(birth_country),
    height_cm = VALUES(height_cm),
    weight_kg = VALUES(weight_kg),
    shoots_catches = VALUES(shoots_catches),
    headshot_url = VALUES(headshot_url)
"""

upsert_skater_query = """
INSERT INTO skater_season_stats (
    player_id, season, team_id, games_played, goals, assists, points,
    plus_minus, penalty_min, shots, avg_toi
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    games_played = VALUES(games_played),
    goals = VALUES(goals),
    assists = VALUES(assists),
    points = VALUES(points),
    plus_minus = VALUES(plus_minus),
    penalty_min = VALUES(penalty_min),
    shots = VALUES(shots),
    avg_toi = VALUES(avg_toi)
"""

player_positions.update({record[0]: record[4] for record in supplemental_player_records})
skater_records = []
goalie_records = []
missing_players = []

for row in skater_data:
    player_id = int(row["player_id"])
    position = player_positions.get(player_id)

    if position is None:
        missing_players.append(player_id)
        continue
    if position == "G":
        goalie_records.append(row)
        continue

    skater_records.append((
        player_id,
        str(row["season"]),
        int(row["team_id"]),
        row.get("games_played"),
        row.get("goals"),
        row.get("assists"),
        row.get("points"),
        row.get("plus_minus"),
        row.get("penalty_min"),
        row.get("shots"),
        row.get("avg_toi")
    ))

if missing_players:
    raise ValueError(f"Could not determine positions for player IDs: {missing_players}")

try:
    if supplemental_player_records:
        skater_cursor.executemany(upsert_player_query, supplemental_player_records)
    if skater_records:
        skater_cursor.executemany(upsert_skater_query, skater_records)
    connection.commit()
except Exception:
    connection.rollback()
    raise

skater_cursor.execute("""
SELECT COUNT(*) AS stats_rows,
       COUNT(DISTINCT player_id) AS player_count,
       COUNT(DISTINCT team_id) AS team_count
FROM skater_season_stats
""")
print(f"Skater records inserted or updated: {len(skater_records)}")
print(f"Goalie records skipped for Table 7: {len(goalie_records)}")
print(f"Supplemental player records added: {len(supplemental_player_records)}")
print(f"avg_toi values present in JSON: {sum(row.get('avg_toi') is not None for row in skater_data)}")
print("Database totals (rows, players, teams):", skater_cursor.fetchone())
skater_cursor.close()

Fetched player details: 10/88
Fetched player details: 20/88
Fetched player details: 30/88
Fetched player details: 40/88
Fetched player details: 50/88
Fetched player details: 60/88
Fetched player details: 70/88
Fetched player details: 80/88
Fetched player details: 88/88
Skater records inserted or updated: 714
Goalie records skipped for Table 7: 0
Supplemental player records added: 88
avg_toi values present in JSON: 0
Database totals (rows, players, teams): (714, 714, 32)


True

In [11]:
df=pd.read_sql_query("SELECT * FROM skater_season_stats", connection)

/var/folders/f7/2dmww_657j51jk3c1xsbw30h0000gn/T/ipykernel_16340/3259520688.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("SELECT * FROM skater_season_stats", connection)


In [12]:
df

,stat_id,player_id,season,team_id,games_played,goals,assists,points,plus_minus,penalty_min,shots,avg_toi
0,1,8470613,20252026,1,82.0,12.0,23.0,35.0,33.0,32.0,172.0,None
1,2,8475172,20252026,1,77.0,16.0,34.0,50.0,-30.0,30.0,213.0,None
2,3,8475754,20252026,1,81.0,33.0,32.0,65.0,15.0,36.0,186.0,None
3,4,8475768,20252026,1,50.0,11.0,15.0,26.0,5.0,8.0,82.0,None
4,5,8476312,20252026,1,79.0,5.0,26.0,31.0,42.0,91.0,128.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...
709,710,8483678,20252026,32,70.0,3.0,7.0,10.0,-16.0,39.0,46.0,None
710,711,8483768,20252026,32,24.0,0.0,3.0,3.0,-12.0,7.0,14.0,None
711,712,8484136,20252026,32,66.0,13.0,6.0,19.0,-13.0,26.0,62.0,None
712,713,8484240,20252026,32,70.0,5.0,16.0,21.0,-23.0,28.0,59.0,None


In [13]:
df.head()

,stat_id,player_id,season,team_id,games_played,goals,assists,points,plus_minus,penalty_min,shots,avg_toi
0,1,8470613,20252026,1,82.0,12.0,23.0,35.0,33.0,32.0,172.0,None
1,2,8475172,20252026,1,77.0,16.0,34.0,50.0,-30.0,30.0,213.0,None
2,3,8475754,20252026,1,81.0,33.0,32.0,65.0,15.0,36.0,186.0,None
3,4,8475768,20252026,1,50.0,11.0,15.0,26.0,5.0,8.0,82.0,None
4,5,8476312,20252026,1,79.0,5.0,26.0,31.0,42.0,91.0,128.0,None


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

SEASON = "20252026"

games_cursor = connection.cursor()
games_cursor.execute("""
CREATE TABLE IF NOT EXISTS games (
    game_id BIGINT PRIMARY KEY,
    season VARCHAR(20),
    game_type INT,
    game_date DATE,
    home_team_id INT NOT NULL,
    away_team_id INT NOT NULL,
    home_score INT,
    away_score INT,
    game_state VARCHAR(20),
    venue_name VARCHAR(150),
    CONSTRAINT fk_games_home_team
        FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    CONSTRAINT fk_games_away_team
        FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB
""")

games_cursor.execute("SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev")
teams_for_schedule = games_cursor.fetchall()
team_id_by_abbrev = {team_abbrev: team_id for team_id, team_abbrev in teams_for_schedule}


def fetch_team_schedule(team_abbrev):
    max_attempts = 5
    for attempt in range(1, max_attempts + 1):
        response = requests.get(
            f"https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/{SEASON}",
            timeout=(5, 10)
        )
        if response.status_code == 429 and attempt < max_attempts:
            time.sleep(2 ** attempt)
            continue
        response.raise_for_status()
        break
    payload = response.json()

    team_records = []
    for game in payload.get("games", []):
        home_abbrev = game["homeTeam"]["abbrev"]
        away_abbrev = game["awayTeam"]["abbrev"]
        home_team_id = team_id_by_abbrev.get(home_abbrev)
        away_team_id = team_id_by_abbrev.get(away_abbrev)
        if home_team_id is None or away_team_id is None:
            continue

        venue = game.get("venue")
        venue_name = venue.get("default") if isinstance(venue, dict) else venue

        team_records.append((
            game["id"],
            str(game.get("season")),
            game.get("gameType"),
            game.get("gameDate"),
            home_team_id,
            away_team_id,
            game["homeTeam"].get("score"),
            game["awayTeam"].get("score"),
            game.get("gameState"),
            venue_name,
        ))

    return team_abbrev, team_records


game_records_by_id = {}
schedule_errors = []

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {}
    for team_id, team_abbrev in teams_for_schedule:
        futures[executor.submit(fetch_team_schedule, team_abbrev)] = team_abbrev
        time.sleep(0.3)

    for completed_count, future in enumerate(as_completed(futures), start=1):
        team_abbrev = futures[future]
        try:
            _, team_records = future.result()
            for record in team_records:
                # dedupe in Python too: each game appears once from each side's schedule
                game_records_by_id[record[0]] = record
            print(f"Fetched {team_abbrev}: {len(team_records)} games ({completed_count}/{len(futures)} teams)")
        except requests.RequestException as error:
            schedule_errors.append((team_abbrev, str(error)))
            print(f"Failed {team_abbrev} ({completed_count}/{len(futures)} teams): {error}")

if schedule_errors:
    raise RuntimeError(
        f"Schedule fetch failed for {len(schedule_errors)} teams; no database rows were written. "
        f"Retry after checking the network: {schedule_errors}"
    )

insert_games_query = """
INSERT IGNORE INTO games (
    game_id, season, game_type, game_date, home_team_id, away_team_id,
    home_score, away_score, game_state, venue_name
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

game_records = list(game_records_by_id.values())
games_cursor.executemany(insert_games_query, game_records)
connection.commit()

games_cursor.execute("SELECT COUNT(*) FROM games")
print(f"Fetched {len(game_records)} unique games across {len(teams_for_schedule)} teams.")
print("Database total (games):", games_cursor.fetchone())
